# Gera imagem Sanduiche

In [1]:
#===================================================================================================#
#                              DEFINE CAMINHO DO DIRETÓRIO
#===================================================================================================#
dir = '/content/'

#===================================================================================================#
#                                      INSIRA A DATA
#===================================================================================================#
# data
ano, mes, dia, hor, min = '2026', '09', '21', '14', '00'

#===================================================================================================#
#                                 BAIXA AS IMAGENS DO CANAL 2 E 13
#===================================================================================================#
# ftp CPTEC
ftp_cptec_abi_10min = 'http://ftp.cptec.inpe.br/goes/'

# ch2
file =  f'{ftp_cptec_abi_10min}goes19/retangular/ch02/{ano}/{mes}/S10165534_{ano}{mes}{dia}{hor}{min}.nc'
!wget -c {file} -P {dir}

# ch13
file =  f'{ftp_cptec_abi_10min}goes19/retangular/ch13/{ano}/{mes}/S10165545_{ano}{mes}{dia}{hor}{min}.nc'
!wget -c {file} -P {dir}

#===================================================================================================#
#                                   INSTALA E OIMPORTA BIBLIOTECAS
#===================================================================================================#
# instala bibliotecas
!pip install -q netCDF4

# Sem essas duas linhas de comando ocorre erro na geracao da figura
import matplotlib as mpl
mpl.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
import sys
from netCDF4 import Dataset as dt
import time
import configparser
import os

#===================================================================================================#
#                                        PROCESSAMENTO
#===================================================================================================#
def reverse_colourmap(cmap, name = 'my_cmap_r'):
    """
    In:
    cmap, name
    Out:
    my_cmap_r

    Explanation:
    t[0] goes from 0 to 1
    row i:   x  y0  y1 -> t[0] t[1] t[2]
                   /
                  /
    row i+1: x  y0  y1 -> t[n] t[1] t[2]

    so the inverse should do the same:
    row i+1: x  y1  y0 -> 1-t[0] t[2] t[1]
                   /
                  /
    row i:   x  y1  y0 -> 1-t[n] t[2] t[1]
    """
    reverse = []
    k = []

    for key in cmap._segmentdata:
        k.append(key)
        channel = cmap._segmentdata[key]
        data = []

        for t in channel:
            data.append((1-t[0],t[2],t[1]))
        reverse.append(sorted(data))

    LinearL = dict(zip(k,reverse))
    my_cmap_r = mpl.colors.LinearSegmentedColormap(name, LinearL)
    return my_cmap_r

# Definicoes iniciais
pscript = sys.argv[1:]
CONF_FILE = pscript[0]     #Arquivo de configuração

# Leitura do arquivo de configurações
config = configparser.ConfigParser()
config.read(CONF_FILE)

# Definição de diretorios
DIROUT = f'{dir}' #config.get(f'{dir}', 'DIROUT')        # Define diretorio de saida NETCDF
DIRLOG = f'{dir}' #config.get(f'{dir}', 'DIRLOG')        # Define diretorio de logs
DIRLOC = f'{dir}' #os.path.dirname(os.path.abspath(__file__))    # Define diretorio local

# Arquivos de entrada
FILEINCH1 = f'{dir}S10165534_{ano}{mes}{dia}{hor}{min}.nc' #pscript[1]		       # Arquivo de entrada CH1
FILEINCH4 = f'{dir}/S10165545_{ano}{mes}{dia}{hor}{min}.nc' #pscript[2]		       # Arquivo de entrada CH4
DIRTMP = f'{dir}' #pscript[3]        # Define diretorio de arquivos temporarios

# Nome dos arquivos de saida
nch13 = os.path.join(DIRTMP, 'ch13.jpg')
nch02 = os.path.join(DIRTMP, 'ch02.jpg')

# Abre arquivo de log
lf = open(os.path.join(DIRLOG, 'imgsnd_' + time.strftime("%Y%m%d")), 'a')

latbounds = [ -56.54, 13.65 ]
lonbounds = [ -82.87, -33.30 ]

# Abrindo arquivo e extraindo variavel CH02
print(FILEINCH1)
ch2_arq = dt(FILEINCH1,mode='r')

# Defining lat/lon subset
lats = ch2_arq.variables['lat'][:]
lons = ch2_arq.variables['lon'][:]
# latitude lower and upper index
latli = np.argmin( np.abs( lats - latbounds[0] ) )
latui = np.argmin( np.abs( lats - latbounds[1] ) )
# longitude lower and upper index
lonli = np.argmin( np.abs( lons - lonbounds[0] ) )
lonui = np.argmin( np.abs( lons - lonbounds[1] ) )
ch2_img = ch2_arq.variables['Band1'][latli:latui , lonli:lonui]
ch2_arq.close()

# Abrindo arquivo e extraindo variavel CH13
ch13_arq = dt(FILEINCH4,mode='r')

# Defining lat/lon subset
lats = ch13_arq.variables['lat'][:]
lons = ch13_arq.variables['lon'][:]
# latitude lower and upper index
latli = np.argmin( np.abs( lats - latbounds[0] ) )
latui = np.argmin( np.abs( lats - latbounds[1] ) )
# longitude lower and upper index
lonli = np.argmin( np.abs( lons - lonbounds[0] ) )
lonui = np.argmin( np.abs( lons - lonbounds[1] ) )
ch13_img = ch13_arq.variables['Band1'][latli:latui , lonli:lonui]
ch13_arq.close()

# Resampled CH13 by a factor of 4 with nearest interpolation:
ch13_img = scipy.ndimage.zoom(ch13_img, 2, order=3)
#ch13_img = scipy.ndimage.zoom(ch13_img, 4, order=3)

# Aplicando conversao
ch13_img = np.flipud(ch13_img)/100
ch2_img = np.flipud(ch2_img)/100

fig = plt.figure(frameon='False')
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)

DPI = 100

npxl_x, npxl_y = np.shape(ch2_img)

# dimensao da figura de acordo com area plotada e resolucao espacial da imagem de satelite
font_size_prop = 256
fig_prop = 1
fsze = npxl_y / font_size_prop

# O tamanho da figura gerada nao sera igual a resolucao espacial real!!!
fig.set_size_inches(npxl_y / float(DPI),npxl_x / float(DPI))

# Definindo valores invalidos
ch13_img[ch13_img > 240] = np.nan

my_cmap_r = reverse_colourmap(mpl.cm.jet)
cm = plt.cm.get_cmap(my_cmap_r)

plt.imshow(ch13_img, cmap=cm, vmin=200., vmax=240.)

# Salvando figura reamostrada
plt.savefig(nch13, dpi=DPI, pad_inches=0)
if os.path.exists(nch13):
    lf.write('>> Arquivo '+nch13+' gerado!')
else:
    print('ERRO: arquivo '+nch13+' nao foi gerado!')

#------------------------------------------------------------------------------------------------------------
reflc_or_tb_max = 105.
reflc_or_tb_min = 0.
gamma_coef = 1.5
ch2_img = (((ch2_img - reflc_or_tb_min) / (reflc_or_tb_max - reflc_or_tb_min)) ** (1.0 / gamma_coef))
#------------------------------------------------------------------------------------------------------------

cm = plt.cm.get_cmap('gray')
plt.imshow(ch2_img, cmap=cm, vmin=np.min(ch2_img), vmax=np.max(ch2_img))

plt.savefig(nch02, dpi=DPI, pad_inches=0)
if os.path.exists(nch02):
    lf.write('>> Arquivo '+nch02+' gerado!')
else:
    print('ERRO: arquivo '+nch02+' nao foi gerado!')

lf.write('> Colab Notebook: Finalizado processamento do arquivo '+nch02+' e '+nch13+' em '+time.strftime("%d/%m/%Y %H:%M") + '\n')
lf.close()

#===================================================================================================#
#                               JUNTA AS IMAGENS DO CH2 E CH13
#===================================================================================================#
# instala o imagemagick
!apt-get install imagemagick

# junta as figuras
!convert {dir}ch13.jpg -alpha On -channel Alpha -evaluate set 70% {dir}ch13.jpg && composite {dir}ch02.jpg {dir}ch13.jpg -compose Multiply -quality 90 {dir}sandwich_resultado.jpg

--2026-09-21 20:19:08--  http://ftp.cptec.inpe.br/goes/goes19/retangular/ch02/2026/09/S10165534_202609211400.nc
Resolving ftp.cptec.inpe.br (ftp.cptec.inpe.br)... 150.163.178.56
Connecting to ftp.cptec.inpe.br (ftp.cptec.inpe.br)|150.163.178.56|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://ftp.cptec.inpe.br/goes/goes19/retangular/ch02/2026/09/S10165534_202609211400.nc [following]
--2026-09-21 20:19:09--  https://ftp.cptec.inpe.br/goes/goes19/retangular/ch02/2026/09/S10165534_202609211400.nc
Connecting to ftp.cptec.inpe.br (ftp.cptec.inpe.br)|150.163.178.56|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57588963 (55M) [application/x-netcdf]
Saving to: ‘/content/S10165534_202609211400.nc’

S10165534_202609211 100%[===================>]  54.92M  13.6MB/s    in 4.0s    

2026-09-21 20:19:14 (13.6 MB/s) - ‘/content/S10165534_202609211400.nc’ saved [57588963/57588963]

--2026-09-21 20:19:14--  http://ftp.cptec.inp

/tmp/ipykernel_1244/2147109848.py:172: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = plt.cm.get_cmap(my_cmap_r)
/tmp/ipykernel_1244/2147109848.py:190: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cm = plt.cm.get_cmap('gray')


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-droid-fallback fonts-noto-mono fonts-urw-base35 ghostscript
  imagemagick-6-common imagemagick-6.q16 libdjvulibre-text libdjvulibre21
  libgs-common libgs10 libgs10-common libidn12 libijs-0.35 libimath-3-1-29t64
  libjbig2dec0 libjxr-tools libjxr0t64 liblqr-1-0 libmagickcore-6.q16-7-extra
  libmagickcore-6.q16-7t64 libmagickwand-6.q16-7t64 libnetpbm11t64
  libopenexr-3-1-30 libraw23t64 libwmflite-0.2-7 netpbm poppler-data
  xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto fonts-freefont-otf | fonts-freefont-ttf fonts-texgyre
  texlive-binaries imagemagick-6-doc autotrace cups-bsd | lpr | lprng enscript
  gimp gnuplot grads hp2xx html2ps libwmf-bin mplayer povray radiance
  sane-utils texlive-base-bin transfig libraw-bin inkscape poppler-utils
  fonts-japanese-mincho | fonts-ipafont-mincho fonts-japanese-gothic
  | fo